# 02 — Preprocessing Pipeline

**Phase 2** of the seismic first-break picking project.

This notebook transforms raw HDF5 seismic data into clean, model-ready
per-shot NPZ files with stratified train/val/test splits.

**EDA-informed decisions locked in this pipeline:**
- Lalor: `resample_poly(x, 1, 2)[:751]` (empirically validated)
- Sudbury: crop to first 751 samples (1500ms)
- Normalization: per-trace max-absolute (ε=1e-10)
- Labels: `SPARE1 > 0` only (FBT/MBT confirmed all-zero)
- Width: variable per-batch, pad to nearest multiple of 16
- Split: per-asset 70/15/15 stratified by median FBT
- Max width cap: 4096 traces (fits T4 GPU)

**Runtime:** ~1-3 hours on Colab CPU depending on Drive I/O speed.
Rerun-safe: skips already-processed shots.

In [ ]:
# Cell 1: Mount Drive and configure paths
import os
import sys
import time

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Drive mounted successfully')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not running on Colab — assuming local execution')

def find_project_root(indicator_file="configs/datasets.yaml"):
    """Dynamically find the project root relative to the notebook location."""
    curr = os.getcwd()
    # Check current and up to 4 parent levels
    for _ in range(5):
        if os.path.exists(os.path.join(curr, indicator_file)):
            return curr
        curr = os.path.dirname(curr)
    # Hardcoded fallback
    return '/content/drive/MyDrive/seismic-first-break-picking'

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Verify structure
assert os.path.isfile('configs/datasets.yaml'), 'datasets.yaml not found'
assert os.path.isfile('configs/preprocessing.yaml'), 'preprocessing.yaml not found'
print(f'Working directory: {os.getcwd()}')
print(f'Python: {sys.version}')

In [ ]:
# Cell 2: Load configuration files
from src.utils.config_loader import load_yaml

project_cfg = load_yaml('configs/datasets.yaml')
preprocess_cfg = load_yaml('configs/preprocessing.yaml')

# Display key parameters
print('=== Harmonization Target ===')
h = preprocess_cfg['harmonization']
print(f"  Sample rate: {h['target_samp_rate_us']} µs ({h['target_samp_rate_us']/1000} ms)")
print(f"  Samples:     {h['target_samp_num']}")
print(f"  Duration:    {h['target_time_window_ms']} ms")

print('\n=== Normalization ===')
n = preprocess_cfg['normalization']
print(f"  Method:  {n['method']}")
print(f"  Epsilon: {n['epsilon']}")

print('\n=== Gather Constraints ===')
g = preprocess_cfg['gather']
print(f"  Min traces:       {g['min_traces']}")
print(f"  Max width cap:    {g['max_width_cap']}")
print(f"  Width divisibility: {g['width_divisibility']}")
print(f"  Batching:         {g['batching_strategy']}")

print('\n=== Split Strategy ===')
s = preprocess_cfg['splitting']
print(f"  Train/Val/Test: {s['train_ratio']}/{s['val_ratio']}/{s['test_ratio']}")
print(f"  Stratify by:    {s['stratify_by']}")
print(f"  Seed:           {s['random_seed']}")

ASSETS = project_cfg['assets']['names']
print(f'\nAssets to process: {ASSETS}')

In [ ]:
# Cell 3: Process all assets — HDF5 → NPZ shot gathers
#
# This is the main pipeline cell. For each asset:
#   1. Opens HDF5, loads metadata, groups by SHOTID
#   2. Sorts traces within each shot by offset
#   3. Harmonizes temporal dimensions (resample Lalor, crop Sudbury)
#   4. Normalizes per-trace (max-abs)
#   5. Validates labels (SPARE1 > 0, range checks, coherence)
#   6. Saves as compressed NPZ
#
# Rerun-safe: skips shots that already have valid NPZ files.
# Expected runtime: ~20-40 min per asset depending on Drive I/O speed.

from src.data.shot_gather_builder import process_asset

processed_dir = os.path.join(PROJECT_ROOT, preprocess_cfg['output']['processed_dir'])
extracted_dir = os.path.join(PROJECT_ROOT, project_cfg['paths']['extracted'])

all_index_rows = []
t_start = time.time()

for asset_name in ASSETS:
    print(f'\n{"="*60}')
    print(f'Processing: {asset_name.upper()}')
    print(f'{"="*60}')

    asset_meta = project_cfg['asset_meta'][asset_name]
    hdf5_path = os.path.join(extracted_dir, asset_meta['extracted_file'])

    if not os.path.isfile(hdf5_path):
        print(f'  ERROR: HDF5 file not found: {hdf5_path}')
        print(f'  Skipping {asset_name}.')
        continue

    rows = process_asset(
        hdf5_path=hdf5_path,
        asset_name=asset_name,
        asset_meta=asset_meta,
        preprocess_cfg=preprocess_cfg,
        output_dir=processed_dir,
    )
    all_index_rows.extend(rows)

elapsed = time.time() - t_start
print(f'\n{"="*60}')
print(f'All assets processed: {len(all_index_rows):,} total shots in {elapsed:.1f}s')
print(f'{"="*60}')

In [ ]:
# Cell 4: Write master index CSV
from src.data.shot_gather_builder import write_master_index

master_path = os.path.join(PROJECT_ROOT, preprocess_cfg['output']['master_index'])
write_master_index(all_index_rows, master_path)

# Quick summary
import pandas as pd
df = pd.read_csv(master_path)
print(f'\nMaster index shape: {df.shape}')
print(f'\nPer-asset summary:')
summary = df.groupby('asset').agg(
    shots=('shot_id', 'count'),
    total_traces=('n_traces', 'sum'),
    labeled_traces=('n_labeled', 'sum'),
    mean_gather_size=('n_traces', 'mean'),
    min_gather_size=('n_traces', 'min'),
    max_gather_size=('n_traces', 'max'),
).round(0)
summary['label_pct'] = (summary['labeled_traces'] / summary['total_traces'] * 100).round(1)
print(summary.to_string())

In [ ]:
# Cell 5: Stratified train/val/test split
from src.data.shot_gather_builder import stratified_split, write_split_index

split_cfg = preprocess_cfg['splitting']

all_index_rows = stratified_split(
    all_index_rows,
    train_ratio=split_cfg['train_ratio'],
    val_ratio=split_cfg['val_ratio'],
    test_ratio=split_cfg['test_ratio'],
    n_bins=split_cfg['n_quantile_bins'],
    seed=split_cfg['random_seed'],
)

split_path = os.path.join(PROJECT_ROOT, preprocess_cfg['output']['split_index'])
write_split_index(all_index_rows, split_path)

In [ ]:
# Cell 6: Generate preprocessing report
from src.data.shot_gather_builder import generate_preprocessing_report

report_path = os.path.join(PROJECT_ROOT, preprocess_cfg['output']['preprocessing_report'])
report = generate_preprocessing_report(all_index_rows, report_path)

# Print summary
import json
print(json.dumps(report, indent=2))

In [ ]:
# Cell 7: Validation — visual spot-check of processed gathers
import numpy as np
import matplotlib.pyplot as plt
from src.data.shot_gather_builder import load_shot_npz

# Load visualization examples for known-good shots
viz_path = os.path.join(PROJECT_ROOT, 'results', 'visualization_examples.json')
with open(viz_path) as f:
    viz_examples = json.load(f)

fig, axes = plt.subplots(3, 4, figsize=(24, 16))
fig.suptitle('Processed Shot Gathers — Visualization Examples', fontsize=16, y=0.98)

for i, ex in enumerate(viz_examples):
    asset = ex['asset']
    shot_id = ex['shot_id']
    difficulty = ex['difficulty']

    npz_path = os.path.join(processed_dir, asset, f'{asset}_{shot_id}.npz')

    if not os.path.isfile(npz_path):
        print(f'WARNING: {npz_path} not found — skipping')
        continue

    data = load_shot_npz(npz_path)
    ax = axes[i // 4, i % 4]

    # Plot gather with first break overlay
    traces = data['traces']  # [751, n_traces]
    vmax = np.percentile(np.abs(traces), 99)
    ax.imshow(traces, aspect='auto', cmap='seismic',
              vmin=-vmax, vmax=vmax, interpolation='none')

    # Overlay labels
    mask = data['label_mask']
    idx = data['labels_idx']
    trace_positions = np.arange(len(mask))
    ax.scatter(trace_positions[mask], idx[mask],
               s=0.5, c='lime', alpha=0.7, zorder=5)

    n_labeled = int(mask.sum())
    ax.set_title(f'{asset} — {difficulty}\nShot {shot_id} | '
                 f'{traces.shape[1]} traces | {n_labeled} labeled',
                 fontsize=9)
    ax.set_ylabel('Sample (time)') if i % 4 == 0 else None
    ax.set_xlabel('Trace') if i >= 8 else None

plt.tight_layout()
save_path = os.path.join(PROJECT_ROOT, 'results', 'preprocessing_validation.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Validation plot saved: {save_path}')

In [ ]:
# Cell 8: Split balance verification
import pandas as pd

split_df = pd.read_csv(split_path)
print('=== Split Counts ===')
split_summary = split_df.groupby(['asset', 'split']).agg(
    n_shots=('shot_id', 'count'),
    total_traces=('n_traces', 'sum'),
    labeled_traces=('n_labeled', 'sum'),
    median_fb=('median_fb_ms', 'median'),
).round(1)
print(split_summary.to_string())

# Check stratification quality: median FBT should be similar across splits
print('\n=== Stratification Quality (median FBT & label fraction per split) ===')
labeled_only = split_df[split_df['median_fb_ms'] > 0]
strat_check = labeled_only.groupby(['asset', 'split']).agg(
    n_gathers=('shot_id', 'count'),
    label_frac_mean=('label_fraction', 'mean'),
    fb_mean=('median_fb_ms', 'mean'),
    fb_median=('median_fb_ms', 'median')
).round(2)
print(strat_check.to_string())

# Overall ratios
print('\n=== Overall Split Ratios ===')
for asset in ASSETS:
    asset_df = split_df[split_df['asset'] == asset]
    total = len(asset_df)
    for split in ['train', 'val', 'test']:
        n = len(asset_df[asset_df['split'] == split])
        pct = n / total * 100 if total > 0 else 0
        print(f'  {asset:12s} {split:5s}: {n:5d} shots ({pct:.1f}%)')

In [ ]:
# Cell 9: DataLoader smoke test
# Quick test to verify the full pipeline from NPZ → batched tensors

try:
    import torch
    from src.data.dataset import ShotGatherDataset, variable_width_collate_fn

    # Load a few val samples
    val_ds = ShotGatherDataset(split_path, split='val')
    print(f'Val dataset: {len(val_ds)} shots')

    # Manual batch collation test
    batch = [val_ds[i] for i in range(min(4, len(val_ds)))]
    collated = variable_width_collate_fn(batch, width_divisibility=16)

    print(f'\nCollated batch shapes:')
    print(f"  traces:     {collated['traces'].shape}")
    print(f"  labels_idx: {collated['labels_idx'].shape}")
    print(f"  label_mask: {collated['label_mask'].shape}")
    print(f"  valid_mask: {collated['valid_mask'].shape}")
    print(f"  n_traces:   {collated['n_traces']}")
    print(f"  assets:     {collated['assets']}")

    # Verify shapes explicitly. A failure here halts the notebook!
    B, C, H, W = collated['traces'].shape
    assert C == 1, f'FATAL: Expected 1 channel, got {C}'
    assert H == 751, f'FATAL: Expected 751 samples (1500ms), got {H}'
    assert W % 16 == 0, f'FATAL: Width {W} not divisible by 16 (U-Net constraint failed)'
    
    # Make sure labels_idx matches width
    assert collated['labels_idx'].shape == (B, W), 'FATAL: label spatial dimensions do not match trace width'

    print(f'\n✓ All shape assertions passed (B={B}, C={C}, H={H}, W={W})')
    print(f'✓ Width divisible by 16: {W} / 16 = {W // 16}')

except AssertionError as e:
    print('\n======================================================')
    print('SMOKE TEST FAILED! HALTING PIPELINE!')
    print('======================================================')
    print(str(e))
    raise  # explicitly fail notebook
except ImportError:
    print('PyTorch not available — skipping DataLoader test')
    print('This is expected on CPU-only Colab. The test will pass during training.')

In [ ]:
# Cell 10: Phase 2 completion summary
import json

print('='*60)
print('PHASE 2 — PREPROCESSING PIPELINE COMPLETE')
print('='*60)

# Count NPZ files per asset
for asset in ASSETS:
    asset_dir = os.path.join(processed_dir, asset)
    if os.path.isdir(asset_dir):
        npz_files = [f for f in os.listdir(asset_dir) if f.endswith('.npz')]
        print(f'  {asset:12s}: {len(npz_files):,} NPZ files')

print(f'\nMaster index: {master_path}')
print(f'Split index:  {split_path}')
print(f'Report:       {report_path}')

# Load and print report totals
with open(report_path) as f:
    rpt = json.load(f)

print(f'\nTotal shots:   {rpt["total"]["n_shots"]:,}')
print(f'Total traces:  {rpt["total"]["n_traces"]:,}')
print(f'Labeled traces: {rpt["total"]["n_labeled"]:,} '
      f'({rpt["total"]["label_fraction"]*100:.1f}%)')

print('\nPHASE 2 PREPROCESSING COMPLETE')